# NeuralMag vs MuMax3 verification (cubic particle)

This notebook reproduces your MuMax setup in NeuralMag, then compares final energies across multiple initial states and particle diameters.

Checks implemented here:
- initialize in different states
- relax/minimize each state
- compare final energies per state
- sweep diameter

If `mumax3` is available on PATH, this notebook can also run MuMax automatically for side-by-side comparison.

In [ ]:
# Optional: ensure required packages in the notebook kernel
# Uncomment if needed:
# %pip install neuralmag pandas matplotlib

In [ ]:
import os
import shutil
import subprocess
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Prevent JAX from grabbing a CUDA device by default (helps avoid GPU OOM in long sweeps).
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import neuralmag as nm

print("NeuralMag backend:", nm.config.backend_name)
print("NeuralMag default device:", nm.config.backend.default_device_str())
print("mumax3 on PATH:", shutil.which("mumax3"))

In [ ]:
# Material parameters from your MuMax snippet
MSAT = 450e3
AEX = 1e-11
ALPHA = 0.1
KC1 = 1.35e4

ANIS1 = np.array([1.0, 1.0, 1.0], dtype=np.float64)
ANIS2 = np.array([1.0, -1.0, 0.0], dtype=np.float64)
ANIS1 /= np.linalg.norm(ANIS1)
ANIS2 /= np.linalg.norm(ANIS2)
ANIS3 = np.cross(ANIS1, ANIS2)
ANIS3 /= np.linalg.norm(ANIS3)

CELLSIZE_M = 1e-9

# Diameter sweep in nm (MuMax script uses D_total as both geometry and grid size)
DIAMETERS_NM = [24, 32, 40, 50]

In [ ]:
def _normalize_vectors(m):
    n = np.linalg.norm(m, axis=-1, keepdims=True)
    return m / np.where(n > 0.0, n, 1.0)


def init_uniform(shape_zyx, vec_xyz):
    nz, ny, nx = shape_zyx
    v = np.asarray(vec_xyz, dtype=np.float32)
    v = v / np.linalg.norm(v)
    m = np.zeros((nz, ny, nx, 3), dtype=np.float32)
    m[...] = v
    return m


def init_vortex(shape_zyx, chirality=1.0, polarity=-1.0, core_radius_px=2.0):
    nz, ny, nx = shape_zyx
    z, y, x = np.meshgrid(
        np.arange(nz, dtype=np.float32),
        np.arange(ny, dtype=np.float32),
        np.arange(nx, dtype=np.float32),
        indexing="ij",
    )
    cx = 0.5 * (nx - 1)
    cy = 0.5 * (ny - 1)
    dx = x - cx
    dy = y - cy
    r2 = dx**2 + dy**2

    mx = -chirality * dy
    my = chirality * dx
    mz = polarity * np.exp(-r2 / (2.0 * core_radius_px**2))

    m = np.stack([mx, my, mz], axis=-1)
    return _normalize_vectors(m).astype(np.float32)


INIT_BUILDERS = {
    "uniform_x": lambda shape: init_uniform(shape, (1.0, 0.0, 0.0)),
    "uniform_111": lambda shape: init_uniform(shape, (1.0, 1.0, 1.0)),
    # MuMax analogue of m = vortex(1, -1)
    "vortex_1_-1": lambda shape: init_vortex(shape, chirality=1.0, polarity=-1.0),
}

In [ ]:
def build_state_cube(diameter_nm, init_m):
    d = int(diameter_nm)
    mesh = nm.Mesh((d, d, d), (CELLSIZE_M, CELLSIZE_M, CELLSIZE_M))
    state = nm.State(mesh)

    state.material.Ms = MSAT
    state.material.A = AEX
    state.material.alpha = ALPHA

    state.material.Kc = KC1
    state.material.Kc_axis1 = ANIS1.tolist()
    state.material.Kc_axis2 = ANIS2.tolist()
    state.material.Kc_axis3 = ANIS3.tolist()

    state.m = nm.VectorCellFunction(state, tensor=state.tensor(init_m.astype(np.float32)))

    nm.ExchangeField().register(state, "exchange")
    nm.DemagField().register(state, "demag")
    nm.CubicAnisotropyField().register(state, "caniso")
    nm.TotalField("exchange", "demag", "caniso").register(state)

    return state


def relax_and_collect_energies(diameter_nm, init_name, relax_tol=2e8, relax_dt=2e-12):
    shape = (int(diameter_nm), int(diameter_nm), int(diameter_nm))
    init_m = INIT_BUILDERS[init_name](shape)

    state = build_state_cube(diameter_nm, init_m)
    e_init = float(nm.config.backend.to_numpy(state.E))

    llg = nm.LLGSolver(state)
    llg.relax(tol=relax_tol, dt=relax_dt)

    rec = {
        "diameter_nm": int(diameter_nm),
        "init_state": init_name,
        "E_init_J": e_init,
        "E_final_J": float(nm.config.backend.to_numpy(state.E)),
    }

    for term in ("exchange", "demag", "caniso"):
        name = f"E_{term}"
        if hasattr(state, name):
            rec[f"{name}_J"] = float(nm.config.backend.to_numpy(getattr(state, name)))

    return rec


records_nm = []
for d_nm in DIAMETERS_NM:
    for init_name in INIT_BUILDERS:
        print(f"NeuralMag: D={d_nm} nm, init={init_name}")
        records_nm.append(relax_and_collect_energies(d_nm, init_name))

nm_df = pd.DataFrame(records_nm).sort_values(["diameter_nm", "init_state"]).reset_index(drop=True)
nm_df

In [ ]:
def mumax_script(diameter_nm, init_mode="vortex_1_-1", load_ovf_path=None):
    if init_mode == "vortex_1_-1":
        init_block = "m = vortex(1, -1)"
    elif init_mode == "uniform_x":
        init_block = "m = uniform(1, 0, 0)"
    elif init_mode == "uniform_111":
        init_block = "m = uniform(1, 1, 1)"
    elif init_mode == "load_ovf":
        if not load_ovf_path:
            raise ValueError("load_ovf mode needs load_ovf_path")
        escaped = str(load_ovf_path).replace('\\', '\\\\')
        init_block = f'm.LoadFile(\"{escaped}\")'
    else:
        raise ValueError(f"Unknown init_mode: {init_mode}")

    return textwrap.dedent(
        f"""
        D_total := {int(diameter_nm)}

        SetGridsize(D_total, D_total, D_total)
        cellsize := 1e-9
        SetCellsize(cellsize, cellsize, cellsize)

        cube := cuboid(D_total*1e-9, D_total*1e-9, D_total*1e-9)

        Msat = 450e3
        Aex  = 1e-11
        alpha = 0.1

        Kc1 = 1.35e4
        anisc1 = vector(1, 1, 1)
        anisc2 = vector(1, -1, 0)

        {init_block}

        minimize()

        tableadd(E_total)
        tablesave()
        """
    ).strip() + "\n"


def run_mumax_case(case_root, diameter_nm, init_mode):
    mumax_bin = shutil.which("mumax3")
    if mumax_bin is None:
        return None

    case_dir = Path(case_root) / f"D{int(diameter_nm):03d}_{init_mode}"
    case_dir.mkdir(parents=True, exist_ok=True)
    mx3 = case_dir / "case.mx3"
    mx3.write_text(mumax_script(diameter_nm, init_mode=init_mode), encoding="utf-8")

    proc = subprocess.run(
        [mumax_bin, str(mx3)],
        cwd=case_dir,
        capture_output=True,
        text=True,
        check=False,
    )
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-2000:])

    table_files = list(case_dir.rglob("table.txt"))
    if not table_files:
        raise FileNotFoundError(f"No MuMax table.txt found under {case_dir}")

    tdf = pd.read_csv(table_files[0], sep="\t", comment="#")
    e_cols = [c for c in tdf.columns if "E_total" in c or c.strip().startswith("E_tot")]
    if not e_cols:
        raise KeyError(f"Could not find E_total column in {table_files[0]}")

    return {
        "diameter_nm": int(diameter_nm),
        "init_state": init_mode,
        "mumax_E_final_J": float(tdf[e_cols[0]].iloc[-1]),
    }

In [ ]:
# Optional MuMax run and comparison
MUMAX_RUN_ROOT = Path("./artifacts/mumax_cube_verification")
records_mumax = []

if shutil.which("mumax3") is None:
    print("mumax3 not found on PATH.")
    print("To enable direct comparison:")
    print("1) install mumax3 and ensure mumax3 is in PATH")
    print("2) rerun this cell")
else:
    for d_nm in DIAMETERS_NM:
        for init_name in INIT_BUILDERS:
            print(f"MuMax3: D={d_nm} nm, init={init_name}")
            records_mumax.append(run_mumax_case(MUMAX_RUN_ROOT, d_nm, init_name))

mumax_df = pd.DataFrame(records_mumax)
mumax_df

In [ ]:
display_cols = ["diameter_nm", "init_state", "E_final_J"]
if not mumax_df.empty:
    compare_df = nm_df.merge(mumax_df, on=["diameter_nm", "init_state"], how="left")
    compare_df["delta_J"] = compare_df["E_final_J"] - compare_df["mumax_E_final_J"]
    display_cols = [
        "diameter_nm",
        "init_state",
        "E_final_J",
        "mumax_E_final_J",
        "delta_J",
    ]
else:
    compare_df = nm_df.copy()

display(compare_df[display_cols].sort_values(["diameter_nm", "init_state"]))

fig, ax = plt.subplots(figsize=(10, 5))
for init_name, g in compare_df.groupby("init_state"):
    ax.plot(g["diameter_nm"], g["E_final_J"], marker="o", label=f"NeuralMag: {init_name}")

if "mumax_E_final_J" in compare_df.columns:
    for init_name, g in compare_df.groupby("init_state"):
        ax.plot(g["diameter_nm"], g["mumax_E_final_J"], marker="x", linestyle="--", label=f"MuMax3: {init_name}")

ax.set_xlabel("Diameter (nm)")
ax.set_ylabel("Final total energy (J)")
ax.set_title("Final energy vs diameter and initial state")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()